# Count users by gender

## What it does

Counts the saved users by the gender of their name. The local AI `llama3.2` decides the gender. The result is only shown.

## What happens to the saved users

It only reads the names. Nothing changes and the gender is not saved.

## Feature it uses

`await countUsersByGender()`, from `logic/count-users-by-gender.js`.

## Failures it can show

- The local AI does not respond → `Could not classify the users`.
- No users → the three numbers are 0.

Ollama must be running (`ollama serve`) with `llama3.2` downloaded (`ollama pull llama3.2`).

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
import Table from "cli-table3";
import { countUsersByGender } from "./logic/count-users-by-gender.js";

try {
  // 1. Count the users by gender: a table with women, men and genderUndefined
  const count = await countUsersByGender();
  const table = new Table({
    head: ["women", "men", "genderUndefined"],
    style: { head: ["cyan", "bold"] },
  });
  table.push([count.women, count.men, count.genderUndefined]);
  console.log(table.toString());
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌───────┬─────┬─────────────────┐
│ women │ men │ genderUndefined │
├───────┼─────┼─────────────────┤
│ 10    │ 11  │ 0               │
└───────┴─────┴─────────────────┘
